In [15]:
import pandas as pd

df = pd.read_csv("../../../Data_LakeHouse/TamAnh_Hospital/Data/data_cleaned_10.csv")

grouped = (
    df.groupby("url")
      .agg({
          "title": "first",
          "heading": lambda x: "\n".join(
              h for h in x.dropna().unique()
          ),
          "content": lambda x: "\n\n".join(
              c for c in x.dropna()
          )
      })
      .reset_index()
)

sample = grouped.sample(10, random_state=42)

In [16]:
def build_document(group):
    title = group["title"].iloc[0]

    sections = []

    for _, row in group.iterrows():
        sections.append(
            f"## {row['heading']}\n\n{row['content']}"
        )

    return pd.Series({
        "title": title,
        "document": f"# {title}\n\n" + "\n\n".join(sections)
    })

documents = (
    df.groupby("url")
      .apply(build_document)
      .reset_index()
)

sample = documents.sample(10, random_state=42)

C:\Users\Windows\AppData\Local\Temp\ipykernel_27724\1911856974.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build_document)


In [17]:
print(sample[["title", "document"]].to_string(index=False, max_colwidth=1000))

                                                                title                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

In [18]:
from pydantic import BaseModel, Field


class QAItem(BaseModel):
    category: str = Field(
        description="One of: fact, multi_chunk, summary"
    )

    question: str = Field(
        description="Question in Vietnamese."
    )

    answer: str = Field(
        description="Ground truth answer."
    )

    evidence: str = Field(
        description="Exact evidence extracted from the document."
    )


class QADataset(BaseModel):
    items: list[QAItem]

In [47]:


QA_GENERATION_PROMPT = """
Bạn là chuyên gia xây dựng benchmark đánh giá RAG.

Nhiệm vụ của bạn là tạo các câu hỏi chất lượng cao từ tài liệu.

Yêu cầu:

- Chỉ sử dụng thông tin có trong tài liệu.
- Không được bổ sung kiến thức bên ngoài.
- Câu trả lời phải chính xác và đầy đủ.
- Evidence phải là đoạn văn hoặc câu trong tài liệu chứng minh đáp án.

Sinh:

- 1 câu hỏi loại fact
- 1 câu hỏi loại multi_chunk
- 1 câu hỏi loại summary

Định nghĩa:

fact:
- Chỉ cần một đoạn nhỏ để trả lời.

multi_chunk:
- Phải kết hợp nhiều phần khác nhau của tài liệu.

summary:
- Yêu cầu tóm tắt về 1 chủ đề cụ thể.

Không tạo câu hỏi mơ hồ.
Không tạo câu hỏi không có đáp án trong tài liệu.
"""


In [48]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    model="qwen3-8b",      # tên model đang serve trên LM Studio
    temperature=0.0,
    timeout=120,
    max_retries=3,
    extra_body={
        "chat_template_kwargs": {
            "enable_thinking": False
        }
    }
)

In [49]:
structured_llm = llm.with_structured_output(QADataset)

In [50]:


def generate_qa(title: str, document: str) -> list[QAItem]:
    result = structured_llm.invoke([
        {
            "role": "system",
            "content": QA_GENERATION_PROMPT
        },
        {
            "role": "user",
            "content": (f"ArithmeticError: {title}\n\n{document}`")
        }
    ])
    return result.items

In [51]:
all_questions = []

for _, row in sample.iterrows():

    try:
        print(f"Processing {row['url']}")
        qa_items = generate_qa(
            row["title"],
            row["document"]
        )
        for qa in qa_items:

            all_questions.append({

                "url": row["url"],

                "title": row["title"],

                "category": qa.category,

                "question": qa.question,

                "ground_truth": qa.answer,

                "evidence": qa.evidence

            })

    except Exception as e:

        print(f"Lỗi {row['url']}: {e}")
        print(f"Tiêu đề: {row['title']}")
        print(f"Tài liệu: {row['document']}")

Processing https://tamanhhospital.vn/cum-mua-nhat-ban/
Processing https://tamanhhospital.vn/chup-x-quang-co-phat-hien-ung-thu-xuong-khong/
Processing https://tamanhhospital.vn/ung-thu-truc-trang-an-hoa-qua-gi/
Processing https://tamanhhospital.vn/cham-soc-benh-nhan-sau-dat-stent-mach-vanh/
Processing https://tamanhhospital.vn/chan-doan-suy-tim/
Processing https://tamanhhospital.vn/sa-tinh-hoan/
Processing https://tamanhhospital.vn/soi-mat-tai-phat-sau-6-lan-mo-lay-soi/
Processing https://tamanhhospital.vn/cach-tang-luong-sua-me/
Processing https://tamanhhospital.vn/viem-duong-ho-hap-nang-vi-o-nhiem-moi-truong/
Processing https://tamanhhospital.vn/benh-viem-co-hoai-tu-qua-trung-gian/


In [53]:
import json

with open("rag_benchmark.jsonl", "w", encoding="utf-8") as f:
    for item in all_questions:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

=======================================================================================================================================================================

In [46]:
########################################################################################################################################################

In [2]:
# ============================================================
# SYNTHESIS GENERATION STRUCTURED OUTPUT
# ============================================================

class SynthesizedEvaluation(BaseModel):
    question: str = Field(
        description="A natural, integrated user question that requires combining information from multiple source pairs."
    )
    ground_truth: str = Field(
        description="A comprehensive, accurate answer to the question, synthesized from the provided sources."
    )
    question_type: str = Field(
        description="Must be 'synthesized'."
    )
    source_ids: list[str] = Field(
        description="List of IDs of the source question-ground_truth pairs that were used/combined to build this question."
    )

class SynthesizedEvaluationList(BaseModel):
    evaluations: list[SynthesizedEvaluation] = Field(
        description="List of synthesized evaluation questions and answers."
    )


# ============================================================
# SYNTHESIS PROMPT
# ============================================================

SYNTHESIS_PROMPT = """
Bạn là chuyên gia xây dựng bộ benchmark đánh giá hệ thống RAG và GraphRAG trong lĩnh vực y tế.

Bạn được cung cấp danh sách các cặp:

- id
- question
- ground_truth

Mỗi cặp đại diện cho một kiến thức nhỏ đã được xác thực.

=========================
MỤC TIÊU
=========================

Hãy tạo khoảng 10 cặp Question + Ground Truth mới có độ khó cao hơn.

Các câu hỏi mới được thiết kế để ĐÁNH GIÁ KHẢ NĂNG TRUY XUẤT TỔNG HỢP của hệ thống GraphRAG và phải gây khó khăn cho Vector Search truyền thống.

=========================
YÊU CẦU
=========================

Mỗi câu hỏi mới PHẢI:

1. Cần ít nhất 2 source_ids để trả lời.

2. Ưu tiên sử dụng 3–6 source_ids nếu phù hợp.

3. Không được tạo câu hỏi mà chỉ cần một đoạn văn hoặc một chunk duy nhất là trả lời được.

4. Câu hỏi phải yêu cầu tổng hợp nhiều thông tin khác nhau.

Ví dụ:

✓ Tổng hợp:
- nguyên nhân
- triệu chứng
- biến chứng
- điều trị
- phòng ngừa
- chẩn đoán
- khi nào cần đi khám
- yếu tố nguy cơ
- tiên lượng

✓ Quan hệ:
- nguyên nhân → hậu quả
- triệu chứng → biến chứng
- bệnh → điều trị
- điều trị → lưu ý
- yếu tố nguy cơ → phòng ngừa

5. Không được chỉ thay đổi cách diễn đạt của câu hỏi gốc.

6. Không được tạo câu hỏi có thể trả lời bằng đúng một ground_truth đã có.

7. Ground Truth phải là phần tổng hợp đầy đủ từ các source_ids đã sử dụng.

8. Không bổ sung kiến thức ngoài dữ liệu được cung cấp.

9. Chỉ sử dụng thông tin có trong các Question và Ground Truth đầu vào.

10. Tất cả đều bằng tiếng Việt.

=========================
ƯU TIÊN CÁC DẠNG CÂU HỎI
=========================

Ưu tiên tạo các dạng:

- Tổng quan về một bệnh
- Tổng hợp nguyên nhân + triệu chứng
- Tổng hợp triệu chứng + điều trị
- Tổng hợp nguyên nhân + biến chứng + phòng ngừa
- Khi nào cần đi khám và các dấu hiệu cảnh báo
- Các yếu tố nguy cơ và cách giảm nguy cơ
- Những lưu ý trong điều trị
- So sánh hai tình trạng hoặc hai phương pháp điều trị (nếu dữ liệu hỗ trợ)
- Mối liên hệ giữa nhiều thông tin khác nhau
- Câu hỏi yêu cầu nhiều bước suy luận

Không tạo:

✗ "Triệu chứng của bệnh A là gì?"

✗ "Nguyên nhân của bệnh B là gì?"

Vì đây chỉ là câu hỏi Local Search.

Thay vào đó tạo:

✓ "Một người có nguy cơ mắc bệnh A sẽ có những dấu hiệu nào, cần đi khám khi nào và điều trị ra sao?"

✓ "Những yếu tố nào làm tăng nguy cơ mắc bệnh, bệnh thường biểu hiện như thế nào và có thể gây biến chứng gì nếu không điều trị?"

=========================
ĐẦU RA
=========================

Đối với mỗi câu hỏi mới, trả về:

- question
- ground_truth
- source_ids

Trong đó:

source_ids phải là danh sách id của các câu hỏi gốc đã sử dụng để tạo câu hỏi.

=========================
DỮ LIỆU ĐẦU VÀO
=========================

"""


In [7]:
# ============================================================
# GENERATE SYNTHESIS BATCH
# ============================================================

async def generate_synthesis_batch(
    batch: list[dict],
    semaphore: asyncio.Semaphore,
    batch_index: int,
) -> list[dict]:
    formatted_list = []
    for item in batch:
        formatted_list.append(
            f"ID: {item['id']}\nQuestion: {item['question']}\nGround Truth: {item['ground_truth']}"
        )
    formatted_pairs = "\n---\n".join(formatted_list)
    
    async with semaphore:
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                response = await client.beta.chat.completions.parse(
                    model=MODEL_NAME,
                    messages=[
                        {
                            "role": "system",
                            "content": SYNTHESIS_PROMPT,
                        },
                        {
                            "role": "user",
                            "content": (f"{formatted_pairs}"),
                        },
                    ],
                    response_format=SynthesizedEvaluationList,
                    temperature=0.3,
                )
                
                parsed = response.choices[0].message.parsed
                if parsed is None or not parsed.evaluations:
                    raise ValueError("LLM returned empty structured output or empty list")
                
                results = []
                for idx, eval_item in enumerate(parsed.evaluations):
                    results.append({
                        "id": f"eval_synth_{batch_index:03d}_{idx+1:03d}",
                        "question": eval_item.question.strip(),
                        "ground_truth": eval_item.ground_truth.strip(),
                        "question_type": eval_item.question_type.strip().lower(),
                        "source_ids": eval_item.source_ids,
                        "metadata": {
                            "batch_index": batch_index,
                            "generation_type": "synthesis"
                        }
                    })
                
                print(f"[SUCCESS] Generated {len(results)} synthesis questions for batch {batch_index}")
                return results
                
            except Exception as e:
                print(f"[RETRY] Batch {batch_index} attempt={attempt}/{MAX_RETRIES} error={e}")
                if attempt < MAX_RETRIES:
                    await asyncio.sleep(2 ** attempt)
        
        print(f"[FAILED] Batch {batch_index}")
        return []


In [8]:
import json
from pathlib import Path


def save_jsonl(
    results,
    output_path,
):
    """
    Save a list of dictionaries or Pydantic models
    to a JSONL file.
    """

    output_path = Path(
        output_path
    )

    # Tạo thư mục nếu chưa tồn tại
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with open(
        output_path,
        "w",
        encoding="utf-8",
    ) as f:

        for item in results:

            # Nếu là Pydantic model
            if hasattr(
                item,
                "model_dump",
            ):
                item = item.model_dump()

            f.write(
                json.dumps(
                    item,
                    ensure_ascii=False,
                )
                + "\n"
            )

    print(
        f"Saved {len(results)} items to "
        f"{output_path}"
    )


In [9]:
# ============================================================
# RUN SYNTHESIS FUNCTION
# ============================================================

async def run_synthesis(
    input_jsonl: str, 
    output_jsonl: str, 
    batch_size: int = 200
):
    print(f"Loading existing data from {input_jsonl}...")
    dataset = []
    with open(input_jsonl, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                dataset.append(json.loads(line.strip()))
                
    print(f"Total loaded questions: {len(dataset)}")
    
    # Chunk dataset into batches of size 200
    batches = [dataset[i:i + batch_size] for i in range(0, len(dataset), batch_size)]
    print(f"Divided into {len(batches)} batches of size {batch_size}")
    
    semaphore = asyncio.Semaphore(1)  # Control concurrency to prevent rate limits
    
    tasks = []
    for idx, batch in enumerate(batches):
        tasks.append(
            generate_synthesis_batch(
                batch=batch,
                semaphore=semaphore,
                batch_index=idx + 1,
            )
        )
        
    batch_results = await asyncio.gather(*tasks)
    
    synthesis_results = []
    for res_list in batch_results:
        synthesis_results.extend(res_list)
        
    print(f"Generated {len(synthesis_results)} synthesis questions in total.")
    
    # Save the synthesis questions
    save_jsonl(synthesis_results, output_jsonl)


In [ ]:
# ============================================================
# EXECUTE SYNTHESIS
# ============================================================

# Để chạy tính năng tổng hợp dữ liệu, hãy bỏ comment dòng dưới đây:
await run_synthesis("evaluation_dataset.jsonl", "evaluation_dataset_synthesis.jsonl", batch_size=100)


Loading existing data from evaluation_dataset.jsonl...
Total loaded questions: 1000
Divided into 10 batches of size 100
[SUCCESS] Generated 5 synthesis questions for batch 1
